## Importing libraries

In [1]:
import magpylib as magpy
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## Initial parameters

In [15]:
resolution=1000    #number of points
R=60e-3            #in m
L=250e-3
layers=5
diameter=1e-3      #diameter of individual wire
turns=20           #number of turns
plotting=1
current=20          #in A

fig = go.Figure()

## Defining Solenoid

In [16]:
def create_solenoid(R,L,layers,diameter,turns,current,plotting):    
    coils=[]
    coordinates=[]
    layer_current=current/layers
    
    for i in range (layers):
    
    
        theta=np.linspace(0,2*np.pi*turns,resolution)
        z=np.linspace(-L/2,L/2,resolution)
        
        x=(R+i*diameter)*np.cos(theta)
        y=(R+i*diameter)*np.sin(theta)
        z=z
        
        
        layer_coordinates=np.stack((x,y,z),axis=1)
        
        coil_layer = magpy.current.Polyline(
            current=layer_current,
            vertices=layer_coordinates
        )
        
        coils.append(coil_layer)
        coordinates=np.vstack((layer_coordinates))

        if plotting==True:
            fig.add_trace(go.Scatter3d(
                x=layer_coordinates[:, 0], y=layer_coordinates[:, 1], z=layer_coordinates[:, 2],
                mode='lines', name=f'Layer {i+1}'))
    
    Solenoid=magpy.Collection(coils)
        
    #Solenoid.show()
    
    
    if plotting==True:
        fig.show(renderer="browser")

    return Solenoid, coordinates

## Cost function

$$\eta = \frac{B_{\max} - B_{\min}}{\bar{B}} \times 10^6 \text{ ppm}$$

Where:
- $B_{\max}$, $B_{\min}$ — max and min field magnitudes over the spherical ROI
- $\bar{B}$ — mean field magnitude over the ROI
- The ROI is a sphere of radius $r \leq \rho R$, sampled on a cubic grid, where $\rho$ is the fractional ROI radius and $R$ is the coil radius

In [17]:
def cost_function(mags,roi): #  of radius in decimal form

    roi_grid=np.mgrid[-roi*R:roi*R:10j, -roi*R:roi*R:10j, -roi*R:roi*R:10j].T
    

    r = np.linalg.norm(roi_grid, axis=-1)   # shape is (N,N,N,3)
    mask = r <= roi*R

    sphere_points = roi_grid[mask]
    
    B_roi=mags.getB(sphere_points)
    
    magnitude_B=np.linalg.norm(B_roi,axis=-1)
    
    mean_B=np.mean(magnitude_B)
    
    B_max=magnitude_B.max()
    B_min=magnitude_B.min()
    eta=((B_max-B_min)/mean_B)*1e6
    
    return eta, mean_B*1000, sphere_points


## Visualize in 3D

In [18]:
def visualize_3d(grid,mags):    #grid to be (N,N,N,3)

    sens = magpy.Sensor(pixel=grid)
    
    # hide axes arrows
    sens.style.arrows.x.show = False
    sens.style.arrows.y.show = False
    sens.style.arrows.z.show = False
    
    # control size
    sens.style.size = 2
    
    # pixel field settings
    sens.style.pixel.field.source = "B"
    sens.style.pixel.field.sizescaling = "linear"
    sens.style.pixel.field.colormap = "Inferno"
    sens.style.pixel.field.colorscaling = "linear"
    sens.style.pixel.field.symbol = "arrow3d"
    
    
    fig = magpy.show([sens, mags], backend="plotly", return_fig=True)
    fig.show(renderer="browser")

## Creating a coil

In [19]:
solenoid,coordinates=create_solenoid(R,L,layers,diameter,turns,current,plotting)  #coils is a mags object
roi=0.1
grid_3D=np.mgrid[-R*1.5:R*1.5:10j, -R*1.5:R*1.5:10j, -L/2:L/2:20j].T
eta,mean_B,sphere_points=cost_function(solenoid,roi)

## Results

In [20]:
visualize_3d(grid_3D,solenoid)

print('For Solenoid','\n')
print(f'homogeneity ',eta,'\n')
print(f'field (mT) ',mean_B,'\n')

For Solenoid 

homogeneity  568.3385596503285 

field (mT)  1.8016925668798474 

